# 03 · Engine sandbox (Phase 3)

Scratch space for the Monte Carlo engine. Each function gets tried here first, then moves into `model/simulate.py` once it works.

Method for every function: **predict the output first, then run it.** Fixed seed so results are reproducible.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
N_RUNS = 10_000
rng = np.random.default_rng(SEED)
DATA = "../data/"   # notebooks/ sits one level below the project root

## Function 1: presets, parameter loader, sampler
**What it does:** holds the three dial presets, loads `parameters.csv`, and draws one coefficient per link, uniform in [low, high].

**My prediction before running:** 29 coefficients, every one inside its own range; 3 presets with the same 6 dial keys.

In [ ]:
PRESETS = {
    "Actual (documented)": {
        "Physical discipline intensity": 9, "Emotional warmth & affection": 3,
        "Autonomy granted": 2, "Normalcy (school peers play)": 2,
        "Body-image comments (negative)": 8, "Parental financial control": 9},
    "Strict coach, warm father": {
        "Physical discipline intensity": 5, "Emotional warmth & affection": 6,
        "Autonomy granted": 4, "Normalcy (school peers play)": 4,
        "Body-image comments (negative)": 4, "Parental financial control": 6},
    "Firm but nurturing (counterfactual)": {
        "Physical discipline intensity": 2, "Emotional warmth & affection": 8,
        "Autonomy granted": 7, "Normalcy (school peers play)": 7,
        "Body-image comments (negative)": 1, "Parental financial control": 4},
}


def load_parameters(path="data/parameters.csv"):
    return pd.read_csv(path)


def sample_params(params, rng):
    """One draw: a coefficient per link, uniform in [low, high]."""
    return dict(zip(params.link_id, rng.uniform(params.low, params.high)))


params = load_parameters(DATA + "parameters.csv")
c = sample_params(params, rng)
inside = all(r.low <= c[r.link_id] <= r.high for r in params.itertuples())
print(len(c), "coefficients; all inside their ranges:", inside)
print({k: round(v, 3) for k, v in list(c.items())[:3]})
print({name: sorted(d) == sorted(PRESETS["Actual (documented)"]) for name, d in PRESETS.items()})

## Function 2: mediators from dials
**What it does:** turns the six dials into six mediators (0-10). Practice is a level; the other five are loads.

**My prediction before running** (Actual preset, range midpoints, worked by hand):
trauma = 10 x (0.60 x 0.9 - 0.40 x 0.3 + 0.25 x 0.8) = **6.2**; practice = 10 x (0.25 + 0.50 x 0.9 + 0.25 x 0.3 + 0.175 x 0.2) = **8.1**; self-image **3.575**; attachment **3.675**; sleep **2.425**; financial delay **4.225**.

In [ ]:
def compute_mediators(dials, c):
    """Dials (0-10) -> six mediators (0-10, clipped). Practice is a LEVEL;
    the other five are LOADS (higher = heavier). Intercepts 0.25 (practice)
    and 0.55 (attachment) are ASSUMPTION baselines."""
    d = {k: v / 10 for k, v in dials.items()}
    trauma = 10 * (c["P01"] * d["Physical discipline intensity"]
                   + c["P02"] * d["Emotional warmth & affection"]
                   + c["P03"] * d["Body-image comments (negative)"])
    practice = 10 * (0.25
                     + c["P04"] * d["Physical discipline intensity"]
                     + c["P05"] * d["Emotional warmth & affection"]
                     + c["P06"] * d["Autonomy granted"])
    self_image = 10 * (c["P07"] * d["Body-image comments (negative)"]
                       + c["P08"] * d["Emotional warmth & affection"])
    attachment = 10 * (0.55
                       + c["P09"] * d["Emotional warmth & affection"]
                       + c["P10"] * d["Autonomy granted"])
    sleep = 10 * (c["P11"] * d["Physical discipline intensity"]
                  + c["P12"] * d["Normalcy (school peers play)"])
    fin_delay = 10 * (c["P13"] * d["Parental financial control"]
                      + c["P14"] * d["Autonomy granted"])
    clip = lambda x: float(np.clip(x, 0, 10))
    return {"Practice hours & skill acquisition": clip(practice),
            "Trauma / fear load": clip(trauma),
            "Self-image & body perception": clip(self_image),
            "Attachment & trust in adults": clip(attachment),
            "Sleep & physical health": clip(sleep),
            "Financial independence timing": clip(fin_delay)}


# Range midpoints: one deterministic "typical draw", so the output can be predicted by hand
params = load_parameters(DATA + "parameters.csv")
mid = dict(zip(params.link_id, (params.low + params.high) / 2))
med = compute_mediators(PRESETS["Actual (documented)"], mid)
{k: round(v, 3) for k, v in med.items()}

## Function 3: career branch node
**What it does:** fame likelihood rises with practice (logistic, midpoint 5.5); trauma splits the fame mass between C (same timeline) and B (later, healthier).

**My prediction before running** (P = 8.1, T = 6.2 from Function 2): fame = 1 / (1 + e^(-2.6/1.2)) = 0.897, so **A = 0.103**, C share = 0.62, **C = 0.556**, **B = 0.341**. Close to the guide's first-run table for Actual (0.119 / 0.333 / 0.548).

In [ ]:
def branch_probabilities(med):
    """Career branches from practice (P) and trauma (T). Logistic midpoint 5.5,
    scale 1.2 and the 0.15-0.9 clip on C's share are ASSUMPTION."""
    P, T = med["Practice hours & skill acquisition"], med["Trauma / fear load"]
    fame = 1 / (1 + np.exp(-(P - 5.5) / 1.2))
    a = 1 - fame
    c_share = float(np.clip(T / 10, 0.15, 0.9))
    return {"A_no_career": a, "B_late_healthy": fame * (1 - c_share),
            "C_same_timeline": fame * c_share}


bp = branch_probabilities(med)
print({k: round(v, 3) for k, v in bp.items()}, "sum =", round(sum(bp.values()), 6))
print("Nurturing preset at midpoints:", {k: round(v, 3) for k, v in branch_probabilities(
    compute_mediators(PRESETS["Firm but nurturing (counterfactual)"], mid)).items()})

## Function 4: outcomes per life stage (with M15)
**What it does:** six outcomes from a 7.0 baseline; loads subtract, practice adds. Relationships now subtracts M15 x trauma, like M10 and M11.

**My prediction before running** (Actual, midpoints, branch C): career = 7 + 0.575 x 3.1 - 0.15 x 4.225 = **8.15**; mental = 7 - 0.6 x 6.2 - 0.375 x 3.575 - 0.325 x 3.675 = **0.75**; relationships = 7 - 0.525 x 3.675 - 0.225 x 4.225 - 0.225 x 6.2 = **2.73**; longevity = 7 - 0.5 x 6.2 - 0.375 x 2.425 - 0.325 x (7 - 0.75) = -1.04, clipped to **0**. Later-stage mental health = 0.75 - 0.5 x 6.2 x 0.3 < 0, so **0**. This is the known floor problem left for Step 3.6.

**What happened:** everything matched except longevity, which came out **0.96**, not 0. The code was right and my arithmetic was wrong: 7 - 3.1 - 0.91 - 2.03 = 0.96. It still sits in the lowest band (<45), which is the same floor problem.

In [ ]:
STAGE_DECAY = {"Childhood": 0.0, "Teen": 0.1, "Young adult": 0.2, "Peak": 0.35, "Later": 0.5}


def compute_outcomes(med, c, branch):
    """Mediators -> six outcomes (0-10, higher = better), per life stage.
    7.0 baseline (ASSUMPTION). Load -> outcome coefficients are positive and
    subtracted; practice is added around a midpoint of 5."""
    m = med
    career = 7.0 + c["M01"] * (m["Practice hours & skill acquisition"] - 5) \
                 - c["M02"] * m["Financial independence timing"]
    if branch == "A_no_career":
        career = min(career, 3.0)          # normal-life ceiling on this dimension
    creative = 7.0 + c["M03"] * (m["Practice hours & skill acquisition"] - 5) \
                   - c["M04"] * m["Trauma / fear load"]
    if branch == "A_no_career":
        creative = min(creative, 4.5)      # music as a gift, not a profession
    mental = 7.0 - c["M05"] * m["Trauma / fear load"] \
                 - c["M06"] * m["Self-image & body perception"] \
                 - c["M07"] * m["Attachment & trust in adults"]
    physical = 7.0 - c["M08"] * m["Self-image & body perception"] \
                   - c["M09"] * m["Sleep & physical health"]
    relations = 7.0 - c["M10"] * m["Attachment & trust in adults"] \
                    - c["M11"] * m["Financial independence timing"] \
                    - c["M15"] * m["Trauma / fear load"]
    longevity = 7.0 - c["M12"] * m["Trauma / fear load"] \
                    - c["M13"] * m["Sleep & physical health"] \
                    - c["M14"] * (7.0 - mental)
    clip = lambda x: float(np.clip(x, 0, 10))
    base = {"Career attainment": clip(career), "Creative output & risk-taking": clip(creative),
            "Mental health": clip(mental), "Physical health & appearance changes": clip(physical),
            "Relationships & family closeness": clip(relations), "Longevity": clip(longevity)}
    staged = {}
    for stage, decay in STAGE_DECAY.items():
        row = dict(base)
        for dim in ("Mental health", "Physical health & appearance changes"):
            row[dim] = clip(row[dim] - decay * m["Trauma / fear load"] * 0.3)
        staged[stage] = row
    return staged  # Longevity is whole-life: the same value at every stage


staged = compute_outcomes(med, mid, "C_same_timeline")
pd.DataFrame(staged).T.round(3)

## Function 5: Monte Carlo runner, bands, longevity bands, sensitivity
**What it does:** samples the coefficients n times per preset, draws a branch per run, and scores outcomes; then summarizes medians with p10-p90 bands, longevity band probabilities, and one-at-a-time sensitivity.

**My prediction before running:** with 2,000 runs, Actual's branch means land near the midpoint values (A ~0.1, C ~0.55); almost all Actual runs fall in the `<45` band because the longevity score sits on the floor; warmth, discipline and body-image comments move mental health most; normalcy and financial control don't move mental health at all (neither feeds it).

In [ ]:
def run_preset(name, dials, params, n=N_RUNS, seed=SEED):
    rng = np.random.default_rng(seed)
    rows, branches = [], []
    for _ in range(n):
        c = sample_params(params, rng)
        med = compute_mediators(dials, c)
        bp = branch_probabilities(med)
        branches.append(bp)
        branch = rng.choice(list(bp), p=list(bp.values()))
        staged = compute_outcomes(med, c, branch)
        for stage, outs in staged.items():
            rows.append({"preset": name, "branch": branch, "stage": stage, **outs})
    return pd.DataFrame(rows), pd.DataFrame(branches)


def p10(s):
    return s.quantile(0.10)


def p90(s):
    return s.quantile(0.90)


DIMS = ["Career attainment", "Creative output & risk-taking", "Mental health",
        "Physical health & appearance changes", "Relationships & family closeness", "Longevity"]


def summarize(df):
    return df.groupby(["preset", "stage"])[DIMS].agg(["median", p10, p90])


# ASSUMPTION: how the 0-10 longevity score maps to lifespan bands. No study gives
# this mapping; it is a stated convention, listed in the assumptions register.
# Step 3.6: was [0, 3, 5, 7, 10]. The score can't exceed the 7.0 baseline, so the
# old top band (7+) was only reachable with zero load; bins now span reachable scores.
LONGEVITY_BINS = [0, 2, 3.5, 5, 10.0001]
LONGEVITY_LABELS = ["<45", "45-55", "55-65", "65+"]


def longevity_bands(df):
    """Share of runs in each lifespan band, per preset. Longevity is whole-life,
    so one stage is enough. Reported as band probabilities, never a single age."""
    whole = df[df["stage"] == "Later"].reset_index(drop=True)
    band = pd.cut(whole["Longevity"], bins=LONGEVITY_BINS, labels=LONGEVITY_LABELS,
                  right=False, include_lowest=True)
    return pd.crosstab(whole["preset"], band, normalize="index").reindex(columns=LONGEVITY_LABELS, fill_value=0.0)


SCORE_SHIFTS = {"career_shift": "Career attainment", "creative_shift": "Creative output & risk-taking",
                "mental_health_shift": "Mental health", "physical_health_shift": "Physical health & appearance changes",
                "relationships_shift": "Relationships & family closeness"}


def sensitivity(dials, params, n=2000, seed=SEED):
    """One-at-a-time: move each dial 2 -> 8 and measure the shift in every outcome dimension.
    Score dimensions: shift in the median (0-10). Longevity (Rule 5: band probabilities only,
    never a score): shift in the share of runs in the <45 and 65+ bands, in percentage points."""
    out = {}
    for dial in dials:
        lo, hi = dict(dials), dict(dials)
        lo[dial], hi[dial] = 2, 8
        d_lo, _ = run_preset("lo", lo, params, n=n, seed=seed)
        d_hi, _ = run_preset("hi", hi, params, n=n, seed=seed)
        row = {k: d_hi[dim].median() - d_lo[dim].median() for k, dim in SCORE_SHIFTS.items()}
        b_lo, b_hi = longevity_bands(d_lo).iloc[0], longevity_bands(d_hi).iloc[0]
        row["longevity_under45_pp"] = 100 * (b_hi["<45"] - b_lo["<45"])
        row["longevity_65plus_pp"] = 100 * (b_hi["65+"] - b_lo["65+"])
        out[dial] = row
    return pd.DataFrame(out).T.sort_values("mental_health_shift", key=abs, ascending=False)


def format_prob(p):
    """Display rule (Step 3.6): never show a band as 0% or 100%. Model shares are
    scenario bands under stated assumptions, not certainties."""
    if p < 0.01:
        return "<1%"
    if p > 0.99:
        return ">99%"
    return f"{p:.0%}"


n_try = 2000
dfs, rows = [], []
for name, dials in PRESETS.items():
    df, bp = run_preset(name, dials, params, n=n_try)
    dfs.append(df); rows.append({"preset": name, **bp.mean().round(3).to_dict()})
res = pd.concat(dfs, ignore_index=True)
display(pd.DataFrame(rows))
display(longevity_bands(res).map(format_prob))
sensitivity(PRESETS["Actual (documented)"], params, n=n_try).round(2).T